> **Superseded 18 Sep 2026.** The debate now runs on a Mac (LM Studio, Qwen3.8) and this workstation (llama.cpp, InternVL), each judge reading its own dataset copy and addressed by Q_ID only -- see `VQA/agent.debate.mc.py` and `VQA/debate.rank.mc.py`. This notebook describes the earlier Colab/Drive design and is kept for reference.


# Agent A - Qwen3.8-27B INT4 on vLLM

One side of the debate driven by `VQA/debate.rank.mc.py`. The orchestrator runs
on the workstation and sends **frame references, not pixels**, so this session
must read the frames itself: that is what `--allowed-local-media-path` does, and
it is why a request is ~1 KB instead of ~800 KB.

Needs an **A100 40GB**; an L4 24GB also fits and, being SM89, can keep an FP8 KV
cache (A100 is SM80 and cannot - see the serve cell).

Run top to bottom, then paste the printed flags into the orchestrator.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# vLLM 0.27 is the only version that registers Qwen3_5ForConditionalGeneration.
# ninja is not optional: FlashInfer shells out to it when it JIT-builds kernels
# on the first real request, and without it the engine dies mid-run.
!pip -q install vllm==0.27.1 ninja


In [ ]:
#@title Session settings
DRIVE_ROOT = "/content/drive/MyDrive/TREC_2026"  #@param {type:"string"}
import secrets
API_KEY = secrets.token_urlsafe(16)
from google.colab import drive
drive.mount("/content/drive")
print("API key for this session:", API_KEY)


In [ ]:
# Cut the debate frames from the videos already on Drive.
# 448x448, mid-point sampling -- the convention VQA/extract.frames.py uses, and
# the same square resize InternVL's own video path does, so the agents see what
# the panel saw, only more of it. ~43k frames, roughly 20-40 min. Resumable.
import json, subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE  = Path(DRIVE_ROOT)
VIDEOS = DRIVE / "videos_TEST_VQA" / "videos"
OUT    = Path("/content/frames448")
manifest = json.load(open(DRIVE / "debate.manifest.json"))["videos"]
print(len(manifest), "videos,", sum(v["n_frames"] for v in manifest.values()), "frames")

def source(stem):
    for name in (stem + ".mp4", stem + ".webm"):
        if (VIDEOS / name).exists():
            return VIDEOS / name
    hits = sorted(VIDEOS.glob(stem + ".*"))
    return hits[0] if hits else None

def cut(item):
    stem, meta = item
    n, duration = int(meta["n_frames"]), float(meta["duration_sec"])
    out = OUT / stem
    if len(list(out.glob("*.jpg"))) == n:          # already cut -> resumable
        return "skip"
    src = source(stem)
    if src is None:
        return "missing"
    out.mkdir(parents=True, exist_ok=True)
    offset = duration / (2 * n)
    vf = "fps=%d/%s:start_time=%.4f,scale=448:448" % (n, duration, offset)
    cmd = ["ffmpeg", "-v", "error", "-an", "-sn", "-i", str(src), "-vf", vf,
           "-q:v", "3", "-frames:v", str(n), str(out / "%03d.jpg"), "-y"]
    subprocess.run(cmd, capture_output=True)
    got = len(list(out.glob("*.jpg")))
    return "ok" if got == n else "short"

with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(cut, manifest.items()))

print(Counter(results))
print("under-cut or missing videos stay at the baseline in the run file")


In [ ]:
import os, subprocess, time, requests
PORT, MODEL = 8011, "RedHatAI/Qwen3.8-27B-INT4"
os.environ["CUDA_HOME"] = os.environ["CUDA_PATH"] = "/usr/local/cuda"

# TRITON_ATTN + bfloat16 KV is the combination measured to work for IMAGE input
# on Ampere: the image-shaped FlashInfer kernel (head_dim 256, FP8 KV) fails to
# build, and Triton refuses FP8 KV below SM89. On an L4 (SM89) you can drop both
# flags and let FlashInfer serve FP8.
cmd = (
    "vllm serve " + MODEL + " --host 0.0.0.0 --port " + str(PORT) +
    " --served-model-name " + MODEL + " --api-key " + API_KEY +
    " --allowed-local-media-path /content/frames448" +
    " --max-model-len 24576 --gpu-memory-utilization 0.92 --enforce-eager" +
    " --limit-mm-per-prompt {\"image\":64} --skip-mm-profiling" +
    " --attention-backend TRITON_ATTN --kv-cache-dtype bfloat16"
)
log = open("/content/vllm.log", "w")
subprocess.Popen(cmd, shell=True, stdout=log, stderr=log)

for _ in range(120):
    try:
        if requests.get("http://localhost:%d/v1/models" % PORT, timeout=5).ok:
            print("server up")
            break
    except Exception:
        pass
    time.sleep(10)
else:
    print(open("/content/vllm.log").read()[-3000:])


In [ ]:
# The FIRST real request is where a broken kernel build shows up -- as an
# engine death, after /v1/models has already answered 200. Find it here, not 300
# queries into the run.
import glob, requests
frame = sorted(glob.glob("/content/frames448/*/001.jpg"))[0]
body = {"model": MODEL, "max_tokens": 32, "messages": [{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": "file://" + frame}},
    {"type": "text", "text": "Reply with one word: what is in this frame?"}]}]}
r = requests.post("http://localhost:%d/v1/chat/completions" % PORT,
                  headers={"Authorization": "Bearer " + API_KEY},
                  json=body, timeout=300)
print(r.status_code, r.text[:300])


In [ ]:
SIDE = "a"
# A public URL for this server. Paste the three flags into the orchestrator on
# the workstation. Quick tunnels are public, which is why the server has a key.
import re, subprocess
subprocess.run("wget -q -O /content/cloudflared "
               "https://github.com/cloudflare/cloudflared/releases/latest/download/"
               "cloudflared-linux-amd64 && chmod +x /content/cloudflared", shell=True)
proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate",
                         "--url", "http://localhost:%d" % PORT],
                        stderr=subprocess.PIPE, text=True)
url = None
for line in proc.stderr:
    hit = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if hit:
        url = hit.group(0)
        break

print("\n" + "=" * 68)
print("  --host-%s %s" % (SIDE, url))
print("  --model-%s %s" % (SIDE, MODEL))
print("  --api-key-%s %s" % (SIDE, API_KEY))
print("=" * 68)
print("Leave this cell running -- closing it drops the tunnel.")
